# LangChain Simple Tool Reference

Developer-facing statements defined in `langchain_core.tools.simple`.

# `Tool: BaseTool`

`Tool` wraps a synchronous function, asynchronous coroutine, or both as a single-input LangChain tool.

## Fields

```python
description: str = "" # Description explaining when the tool should be used
func: Callable[..., str] | None # Synchronous function executed by the tool
coroutine: Callable[..., Awaitable[str]] | None = None # Asynchronous function executed by the tool
```

## Constructor

```python
Tool(
    name: str, # Unique tool name
    func: Callable[..., Any] | None, # Synchronous function executed by the tool
    description: str, # Tool purpose and usage description
    *,
    coroutine: Callable[..., Awaitable[Any]] | None = None, # Asynchronous function executed by the tool
    args_schema: ArgsSchema | None = None, # Optional input-validation schema
    return_direct: bool = False, # Whether agent execution stops after this tool
    **kwargs: Any, # Additional BaseTool fields
) -> None # Initialize the tool
```

## Overridden Properties and Methods

### `args`

Returns the configured argument schema.

When no `args_schema` is supplied, it returns one string argument named `tool_input`.

### `ainvoke`

Uses the configured coroutine for asynchronous execution.

When no coroutine is configured, it runs the synchronous implementation in an executor.

### `_run`

Executes `func` synchronously.

Raises `NotImplementedError` when no synchronous function is configured.

### `_arun`

Executes `coroutine` asynchronously.

## Class Method

### `from_function`

Creates a `Tool` from a synchronous function, asynchronous coroutine, or both.

```python
Tool.from_function(
    func: Callable[..., Any] | None, # Synchronous function executed by the tool
    name: str, # Unique tool name
    description: str, # Tool purpose and usage description
    return_direct: bool = False, # Whether agent execution stops after this tool
    args_schema: ArgsSchema | None = None, # Optional input-validation schema
    coroutine: Callable[..., Awaitable[Any]] | None = None, # Asynchronous function executed by the tool
    **kwargs: Any, # Additional Tool fields
) -> Tool # Return the generated tool
```

Raises `ValueError` when both `func` and `coroutine` are `None`.

## Behaviour

- Accepts exactly one input value.
- Raises `ToolException` when multiple input values are supplied.
- Use `StructuredTool` for functions requiring multiple independent arguments.
- Passes child callbacks when the wrapped function accepts a `callbacks` parameter.
- Passes `RunnableConfig` when the wrapped function declares a supported config parameter.

In [ ]:
from langchain_core.tools import Tool # Import the single-input Tool class


def uppercase_text(text: str) -> str: # Define the synchronous function used by the first tool
    return text.upper() # Return the supplied text in uppercase


def count_characters(text: str) -> str: # Define the function used by the second tool
    return f"Character count: {len(text)}" # Return the number of characters as text


uppercase_tool: Tool = Tool( # Create a Tool directly through its constructor
    name="uppercase_text", # Set the unique tool name
    func=uppercase_text, # Set the synchronous function executed by the tool
    description="Convert the supplied text to uppercase", # Describe the tool's purpose
) # Finish creating the first tool


character_count_tool: Tool = Tool.from_function( # Create another Tool using the class method
    func=count_characters, # Set the synchronous function executed by the tool
    name="count_characters", # Set the unique tool name
    description="Count the characters in the supplied text", # Describe the tool's purpose
) # Finish creating the second tool


uppercase_result: str = uppercase_tool.invoke("hello langchain") # Invoke the first single-input tool

count_result: str = character_count_tool.invoke("LangChain") # Invoke the second single-input tool


print("Uppercase result:", uppercase_result) # Display the uppercase result

print("Character-count result:", count_result) # Display the character-count result

print("Tool name:", uppercase_tool.name) # Display the first tool's name

print("Tool arguments:", uppercase_tool.args) # Display its default single-input schema